# Validate `movies_final.csv` / `ratings_final.csv`

In [ ]:
import sys
import os
sys.path.append("/Users/aryanshah/Developer/Thesis/recosys/src")   

import pandas as pd
import numpy as np

from mappings import (
    CANONICAL_GENRES, UNKNOWN_GENRE_LABEL,
    CANONICAL_LANGUAGE_MAP, UNKNOWN_LANGUAGE_LABEL,
)

MOVIES_PATH = (
    "/Users/aryanshah/Developer/Thesis/recosys/data/processed/movies_final.csv"
)
RATINGS_PATH = (
    "/Users/aryanshah/Developer/Thesis/recosys/data/processed/ratings_final.csv"
)

movies = pd.read_csv(MOVIES_PATH)
ratings = pd.read_csv(RATINGS_PATH)

print("movies_final:", movies.shape)
print("ratings_final:", ratings.shape)
movies.head()

## 1. Schema

In [ ]:
REQUIRED_MOVIE_COLUMNS = ["movieId","tmdbId","title","ml_genres","clean_genres",
                           "language","overview","release_year","vote_average",
                           "vote_count","popularity"]

missing_cols = [c for c in REQUIRED_MOVIE_COLUMNS if c not in movies.columns]
print("Missing columns:", missing_cols or "None")
movies.dtypes

## 2. Nulls / duplicates — movies_final

In [ ]:
dup_movie_ids = movies["movieId"].duplicated().sum()
null_movie_ids = movies["movieId"].isna().sum()
empty_genres = (movies["clean_genres"] == "").sum()
empty_lang = (movies["language"] == "").sum()

print("Duplicate movieId rows:", dup_movie_ids)
print("Null movieId:", null_movie_ids)
print("Empty clean_genres:", empty_genres)
print("Empty language:", empty_lang)
print()
movies.isna().sum()

In [ ]:
tmdb_raw = pd.read_csv(
    "/Users/aryanshah/Developer/Thesis/recosys/data/raw/tmdb/movies_metadata.csv",
    low_memory=False,
)
tmdb_raw["original_language"].value_counts().head(20)


## 3. Genre vocabulary

In [ ]:
valid_genre_tokens = set(CANONICAL_GENRES) | {UNKNOWN_GENRE_LABEL}

all_tokens = set()
for g in movies["clean_genres"].dropna():
    all_tokens.update(g.split("|"))

invalid_genre_tokens = all_tokens - valid_genre_tokens
print("Invalid genre tokens:", invalid_genre_tokens or "None — all canonical")

genre_counts = pd.Series(
    [t for g in movies["clean_genres"].dropna() for t in g.split("|")]
).value_counts()
genre_counts

## 4. Language vocabulary

In [ ]:
from mappings import MISSING_LANGUAGE_LABEL

valid_langs = set(CANONICAL_LANGUAGE_MAP.values()) | {
    UNKNOWN_LANGUAGE_LABEL,
    MISSING_LANGUAGE_LABEL,
}
invalid_langs = set(movies["language"].dropna().unique()) - valid_langs
print("Invalid language labels:", invalid_langs or "None — all canonical")

movies["language"].value_counts()

## 5. release_year sanity

In [ ]:
print("release_year range:", movies["release_year"].min(), "-", movies["release_year"].max())
missing_year_pct = movies["release_year"].isna().mean()
print(f"Missing release_year: {movies['release_year'].isna().sum()} ({missing_year_pct:.1%})")

bad_years = movies[(movies["release_year"] < 1888) | (movies["release_year"] > 2026)]
print("Implausible years (<1888 or >2026):", len(bad_years))

## 6. overview coverage

In [ ]:
empty_overview = (movies["overview"] == "").sum()
print(f"Empty overview: {empty_overview} ({empty_overview/len(movies):.1%})")

## 7. ratings_final — validity

In [ ]:
bad_rating_range = (~ratings["rating"].between(0.5, 5.0)).sum()
null_user = ratings["userId"].isna().sum()
null_movie = ratings["movieId"].isna().sum()
dup_pairs = ratings.duplicated(subset=["userId","movieId"]).sum()

print("Ratings outside [0.5, 5.0]:", bad_rating_range)
print("Null userId:", null_user)
print("Null movieId:", null_movie)
print("Duplicate (userId, movieId) pairs:", dup_pairs)

## 8. Orphan check
Every movieId in ratings must exist in movies_final — otherwise SVD trains on IDs the CBF/hybrid can't map back to metadata.

In [ ]:
valid_ids = set(movies["movieId"])
orphans = ratings.loc[~ratings["movieId"].isin(valid_ids)]
print("Orphan ratings:", len(orphans))

## 9. min-activity threshold actually holds
clean_data.py filters at >=20/>=20 iteratively — confirm nothing below that slipped through.

In [ ]:
user_counts = ratings["userId"].value_counts()
movie_counts = ratings["movieId"].value_counts()

users_below = (user_counts < 20).sum()
movies_below = (movie_counts < 20).sum()

print("Users with <20 ratings:", users_below)
print("Movies with <20 ratings:", movies_below)
print("Unique users:", ratings['userId'].nunique(), "| Unique movies:", ratings['movieId'].nunique())

## 10. Genre & language distribution (thesis-ready plots)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
genre_counts.head(15).plot(kind="bar", ax=axes[0], title="Genre distribution (movies_final)")
movies["language"].value_counts().head(10).plot(kind="bar", ax=axes[1], title="Language distribution (movies_final)")
plt.tight_layout()
plt.show()

## 11. Summary — PASS/FAIL

In [ ]:
checks = {
    "No missing required columns": len(missing_cols) == 0,
    "movieId unique in movies_final": dup_movie_ids == 0,
    "No null movieId": null_movie_ids == 0,
    "No empty clean_genres": empty_genres == 0,
    "No empty language": empty_lang == 0,
    "All genre tokens canonical": len(invalid_genre_tokens) == 0,
    "All language labels canonical": len(invalid_langs) == 0,
    "No implausible release_year": len(bad_years) == 0,
    "Ratings within [0.5, 5.0]": bad_rating_range == 0,
    "No null userId/movieId in ratings": null_user == 0 and null_movie == 0,
    "No duplicate rating pairs": dup_pairs == 0,
    "No orphan ratings": len(orphans) == 0,
    "Min-activity threshold (>=20) holds": users_below == 0 and movies_below == 0,
}

for name, passed in checks.items():
    print(("PASS" if passed else "FAIL"), "-", name)

print()
n_fail = sum(not v for v in checks.values())
print("ALL CHECKS PASSED" if n_fail == 0 else f"{n_fail} CHECK(S) FAILED — see above")